# Notebook 01: Manufacturing Sensor Data Exploration & Baselines

**ManufacturingAgent: Evidence-Grounded Decision Support**

This notebook inspects the machine fleet telemetry database, analyzes sensor parameter distributions (Temperature, Vibration RMS, Pressure, Speed, Humidity), and establishes engineering baseline thresholds.


In [1]:
import sys
import os
import json
import pandas as pd

# Ensure backend is in path
sys.path.insert(0, os.path.abspath('..'))

# Load machine fleet database
with open('../data/machines.json', 'r') as f:
    data = json.load(f)

machines = data['machines']
print(f'Total Fleet Machines: {len(machines)}')
for m_id, info in machines.items():
    print(f"{m_id}: {info['model']} at {info['location']} - Status: {info['current_sensors']['status']}")


Total Fleet Machines: 4
M-101: CNC-Mill-V4 at Bay-A-Line-1 - Status: RUNNING
M-102: CNC-Mill-V4 at Bay-A-Line-2 - Status: WARN_TELEMETRY
M-201: Lathe-Pro-X at Bay-B-Line-1 - Status: ELEVATED_RISK
M-301: CNC-Mill-V4 at Bay-C-Line-1 - Status: SENSOR_FAULT


## Extract Telemetry Dataframe


In [2]:
records = []
for m_id, info in machines.items():
    row = {'machine_id': m_id, 'model': info['model'], 'location': info['location']}
    row.update(info['current_sensors'])
    records.append(row)

df = pd.DataFrame(records)
df


,machine_id,model,location,temperature,vibration,pressure,speed,humidity,status
0,M-101,CNC-Mill-V4,Bay-A-Line-1,52.4,0.85,5.4,4500,45.0,RUNNING
1,M-102,CNC-Mill-V4,Bay-A-Line-2,74.8,2.65,4.1,6200,52.0,WARN_TELEMETRY
2,M-201,Lathe-Pro-X,Bay-B-Line-1,89.6,5.42,2.9,8200,48.0,ELEVATED_RISK
3,M-301,CNC-Mill-V4,Bay-C-Line-1,-999.0,0.00,-1.0,0,0.0,SENSOR_FAULT


## Historical Telemetry Trends for M-101 and M-201


In [3]:
history_records = []
for m_id, info in machines.items():
    for h in info.get('history', []):
        h_row = {'machine_id': m_id, 'model': info['model']}
        h_row.update(h)
        history_records.append(h_row)

hdf = pd.DataFrame(history_records)
hdf


,machine_id,model,timestamp,temperature,vibration,pressure,status,event
0,M-101,CNC-Mill-V4,2026-08-20T08:00:00Z,50.1,0.80,5.4,RUNNING,Routine start
1,M-101,CNC-Mill-V4,2026-08-22T14:30:00Z,53.0,0.88,5.5,RUNNING,Batch finishing
2,M-101,CNC-Mill-V4,2026-08-24T11:15:00Z,52.4,0.85,5.4,RUNNING,Tool change completed
3,M-102,CNC-Mill-V4,2026-08-23T09:00:00Z,69.2,2.10,4.3,WARN_TELEMETRY,Thermal rise detected
4,M-102,CNC-Mill-V4,2026-08-24T15:00:00Z,73.5,2.45,4.2,WARN_TELEMETRY,Edge vibration drift
5,M-102,CNC-Mill-V4,2026-08-25T10:00:00Z,74.8,2.65,4.1,WARN_TELEMETRY,Ongoing drift
6,M-201,Lathe-Pro-X,2026-08-24T18:00:00Z,78.0,3.40,3.5,WARN_TELEMETRY,Rapid thermal gradient
7,M-201,Lathe-Pro-X,2026-08-25T04:00:00Z,85.2,4.80,3.1,ELEVATED_RISK,Critical vibration excursion
8,M-201,Lathe-Pro-X,2026-08-25T08:00:00Z,89.6,5.42,2.9,ELEVATED_RISK,Severe dual fault breach
9,M-301,CNC-Mill-V4,2026-08-25T07:30:00Z,48.0,0.70,5.2,RUNNING,Normal baseline
